In [70]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import (
    LabelEncoder,
    OneHotEncoder,
    StandardScaler,
    MinMaxScaler
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.model_selection import train_test_split

In [71]:
df = pd.read_csv("../data/telecom_churn_clean.csv")
print(df.head())

   customer_id telecom_partner gender  age              state     city  \
0            1    Reliance Jio      F   25          Karnataka  Kolkata   
1            2    Reliance Jio      F   55            Mizoram   Mumbai   
2            3        Vodafone      F   57  Arunachal Pradesh    Delhi   
3            4            BSNL      M   46         Tamil Nadu  Kolkata   
4            5            BSNL      F   26            Tripura    Delhi   

   pincode date_of_registration  num_dependents  estimated_salary  calls_made  \
0   755597           2020-01-01               4            124962          44   
1   125926           2020-01-01               2            130556          62   
2   423976           2020-01-01               0            148828          49   
3   522841           2020-01-01               1             38722          80   
4   740247           2020-01-01               2             55098          78   

   sms_sent  data_used  churn  
0        45          0  False  
1   

In [72]:
df = df.drop(columns=["customer_id","pincode"])
print(df.head())

  telecom_partner gender  age              state     city  \
0    Reliance Jio      F   25          Karnataka  Kolkata   
1    Reliance Jio      F   55            Mizoram   Mumbai   
2        Vodafone      F   57  Arunachal Pradesh    Delhi   
3            BSNL      M   46         Tamil Nadu  Kolkata   
4            BSNL      F   26            Tripura    Delhi   

  date_of_registration  num_dependents  estimated_salary  calls_made  \
0           2020-01-01               4            124962          44   
1           2020-01-01               2            130556          62   
2           2020-01-01               0            148828          49   
3           2020-01-01               1             38722          80   
4           2020-01-01               2             55098          78   

   sms_sent  data_used  churn  
0        45          0  False  
1        39       5973  False  
2        24        193   True  
3        25       9377   True  
4        15       1393  False  


In [73]:
print(df.dtypes)

telecom_partner           str
gender                    str
age                     int64
state                     str
city                      str
date_of_registration      str
num_dependents          int64
estimated_salary        int64
calls_made              int64
sms_sent                int64
data_used               int64
churn                    bool
dtype: object


In [74]:
df["date_of_registration"] = pd.to_datetime(df["date_of_registration"])


In [75]:
print(df.dtypes)

telecom_partner                    str
gender                             str
age                              int64
state                              str
city                               str
date_of_registration    datetime64[us]
num_dependents                   int64
estimated_salary                 int64
calls_made                       int64
sms_sent                         int64
data_used                        int64
churn                             bool
dtype: object


In [76]:
reference_date = df["date_of_registration"].max()
df["customer_tenure"] = (reference_date - df["date_of_registration"]).dt.days

df["year_of_registration"] = df["date_of_registration"].dt.year
df["month_of_registration"] = df["date_of_registration"].dt.month

df = df.drop(columns=["date_of_registration"])

print(df.head())

  telecom_partner gender  age              state     city  num_dependents  \
0    Reliance Jio      F   25          Karnataka  Kolkata               4   
1    Reliance Jio      F   55            Mizoram   Mumbai               2   
2        Vodafone      F   57  Arunachal Pradesh    Delhi               0   
3            BSNL      M   46         Tamil Nadu  Kolkata               1   
4            BSNL      F   26            Tripura    Delhi               2   

   estimated_salary  calls_made  sms_sent  data_used  churn  customer_tenure  \
0            124962          44        45          0  False             1219   
1            130556          62        39       5973  False             1219   
2            148828          49        24        193   True             1219   
3             38722          80        25       9377   True             1219   
4             55098          78        15       1393  False             1219   

   year_of_registration  month_of_registration  
0      

In [77]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 243553 entries, 0 to 243552
Data columns (total 14 columns):
 #   Column                 Non-Null Count   Dtype
---  ------                 --------------   -----
 0   telecom_partner        243553 non-null  str  
 1   gender                 243553 non-null  str  
 2   age                    243553 non-null  int64
 3   state                  243553 non-null  str  
 4   city                   243553 non-null  str  
 5   num_dependents         243553 non-null  int64
 6   estimated_salary       243553 non-null  int64
 7   calls_made             243553 non-null  int64
 8   sms_sent               243553 non-null  int64
 9   data_used              243553 non-null  int64
 10  churn                  243553 non-null  bool 
 11  customer_tenure        243553 non-null  int64
 12  year_of_registration   243553 non-null  int32
 13  month_of_registration  243553 non-null  int32
dtypes: bool(1), int32(2), int64(7), str(4)
memory usage: 22.5 MB
None


In [78]:
for col in ["telecom_partner","gender","state","city"]:
    print(df[col].nunique())

print(df["city"].unique())

4
2
28
6
<StringArray>
['Kolkata', 'Mumbai', 'Delhi', 'Chennai', 'Hyderabad', 'Bangalore']
Length: 6, dtype: str


In [79]:
df["gender"] = df["gender"].map({"M":0,"F":1})
print(df.head())

  telecom_partner  gender  age              state     city  num_dependents  \
0    Reliance Jio       1   25          Karnataka  Kolkata               4   
1    Reliance Jio       1   55            Mizoram   Mumbai               2   
2        Vodafone       1   57  Arunachal Pradesh    Delhi               0   
3            BSNL       0   46         Tamil Nadu  Kolkata               1   
4            BSNL       1   26            Tripura    Delhi               2   

   estimated_salary  calls_made  sms_sent  data_used  churn  customer_tenure  \
0            124962          44        45          0  False             1219   
1            130556          62        39       5973  False             1219   
2            148828          49        24        193   True             1219   
3             38722          80        25       9377   True             1219   
4             55098          78        15       1393  False             1219   

   year_of_registration  month_of_registration  
0

In [ ]:
# df = pd.get_dummies(df,columns=["telecom_partner","state","city"],drop_first=True,dtype=int)
# print(df.head())
df.groupby("state")["churn"].mean().sort_values()


state
West Bengal          0.194004
Punjab               0.194251
Chhattisgarh         0.194307
Tripura              0.194804
Kerala               0.195301
Bihar                0.196340
Meghalaya            0.197490
Andhra Pradesh       0.198520
Arunachal Pradesh    0.198927
Maharashtra          0.199093
Tamil Nadu           0.199358
Goa                  0.199563
Gujarat              0.200232
Haryana              0.200343
Sikkim               0.200714
Uttar Pradesh        0.200755
Telangana            0.201059
Manipur              0.201133
Rajasthan            0.202171
Nagaland             0.202289
Odisha               0.202479
Assam                0.202995
Madhya Pradesh       0.203811
Himachal Pradesh     0.204100
Uttarakhand          0.204155
Mizoram              0.206468
Karnataka            0.207123
Jharkhand            0.211194
Name: churn, dtype: float64

In [83]:
df.groupby("city")["churn"].mean().sort_values()

city
Delhi        0.197499
Chennai      0.197551
Bangalore    0.200502
Kolkata      0.201569
Mumbai       0.201580
Hyderabad    0.204162
Name: churn, dtype: float64

In [84]:
df.groupby("telecom_partner")["churn"].mean().sort_values()

telecom_partner
BSNL            0.198607
Vodafone        0.199484
Reliance Jio    0.200154
Airtel          0.203661
Name: churn, dtype: float64

In [85]:
df["telecom_partner"].value_counts()

telecom_partner
Reliance Jio    61123
Airtel          60905
Vodafone        60802
BSNL            60723
Name: count, dtype: int64

In [87]:
print(df.info())


<class 'pandas.DataFrame'>
RangeIndex: 243553 entries, 0 to 243552
Data columns (total 14 columns):
 #   Column                 Non-Null Count   Dtype
---  ------                 --------------   -----
 0   telecom_partner        243553 non-null  str  
 1   gender                 243553 non-null  int64
 2   age                    243553 non-null  int64
 3   state                  243553 non-null  str  
 4   city                   243553 non-null  str  
 5   num_dependents         243553 non-null  int64
 6   estimated_salary       243553 non-null  int64
 7   calls_made             243553 non-null  int64
 8   sms_sent               243553 non-null  int64
 9   data_used              243553 non-null  int64
 10  churn                  243553 non-null  bool 
 11  customer_tenure        243553 non-null  int64
 12  year_of_registration   243553 non-null  int32
 13  month_of_registration  243553 non-null  int32
dtypes: bool(1), int32(2), int64(8), str(3)
memory usage: 22.5 MB
None


In [89]:
df = pd.get_dummies(df,columns=["telecom_partner","state","city"],drop_first=True,dtype=int)
print(df.head())

   gender  age  num_dependents  estimated_salary  calls_made  sms_sent  \
0       1   25               4            124962          44        45   
1       1   55               2            130556          62        39   
2       1   57               0            148828          49        24   
3       0   46               1             38722          80        25   
4       1   26               2             55098          78        15   

   data_used  churn  customer_tenure  year_of_registration  ...  \
0          0  False             1219                  2020  ...   
1       5973  False             1219                  2020  ...   
2        193   True             1219                  2020  ...   
3       9377   True             1219                  2020  ...   
4       1393  False             1219                  2020  ...   

   state_Telangana  state_Tripura  state_Uttar Pradesh  state_Uttarakhand  \
0                0              0                    0                  0  

In [91]:
# df = df.drop(columns=["state","city","telecom_partner"])
# print(df.head())

print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 243553 entries, 0 to 243552
Data columns (total 46 columns):
 #   Column                        Non-Null Count   Dtype
---  ------                        --------------   -----
 0   gender                        243553 non-null  int64
 1   age                           243553 non-null  int64
 2   num_dependents                243553 non-null  int64
 3   estimated_salary              243553 non-null  int64
 4   calls_made                    243553 non-null  int64
 5   sms_sent                      243553 non-null  int64
 6   data_used                     243553 non-null  int64
 7   churn                         243553 non-null  bool 
 8   customer_tenure               243553 non-null  int64
 9   year_of_registration          243553 non-null  int32
 10  month_of_registration         243553 non-null  int32
 11  telecom_partner_BSNL          243553 non-null  int64
 12  telecom_partner_Reliance Jio  243553 non-null  int64
 13  telecom_partner_Vodafone 

In [92]:
churn = df.pop("churn")
df["churn"] = churn

print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 243553 entries, 0 to 243552
Data columns (total 46 columns):
 #   Column                        Non-Null Count   Dtype
---  ------                        --------------   -----
 0   gender                        243553 non-null  int64
 1   age                           243553 non-null  int64
 2   num_dependents                243553 non-null  int64
 3   estimated_salary              243553 non-null  int64
 4   calls_made                    243553 non-null  int64
 5   sms_sent                      243553 non-null  int64
 6   data_used                     243553 non-null  int64
 7   customer_tenure               243553 non-null  int64
 8   year_of_registration          243553 non-null  int32
 9   month_of_registration         243553 non-null  int32
 10  telecom_partner_BSNL          243553 non-null  int64
 11  telecom_partner_Reliance Jio  243553 non-null  int64
 12  telecom_partner_Vodafone      243553 non-null  int64
 13  state_Arunachal Pradesh  

In [94]:
df["churn"] = df["churn"].map({True:1,False:0})

In [96]:
print(df["churn"])

0         0
1         0
2         1
3         1
4         0
         ..
243548    0
243549    0
243550    0
243551    0
243552    0
Name: churn, Length: 243553, dtype: int64


In [97]:
df.to_csv("../data/feature_engineered_telecom_churn.csv",index=False)